In [ ]:
# =============================
# 0. Imports
# =============================
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.impute import KNNImputer          # <- add this
from xgboost import XGBClassifier

# =============================
# 1. Load data
# =============================
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

print("Train shape:", train.shape)
print("Test shape:", test.shape)

# =============================
# 2. Feature / target setup
# =============================
feature_cols = [c for c in train.columns if c.startswith("Attr")]
target_col = "class"

X_full_raw = train[feature_cols].values
y_full = train[target_col].values

X_test_raw = test[feature_cols].values

# =============================
# 3. Winsorizer definition
# =============================
class Winsorizer(BaseEstimator, TransformerMixin):
    def __init__(self, lower=0.5, upper=99.5):
        self.lower = lower
        self.upper = upper

    def fit(self, X, y=None):
        X = np.asarray(X, dtype=float)
        self.lower_bounds_ = np.nanpercentile(X, self.lower, axis=0)
        self.upper_bounds_ = np.nanpercentile(X, self.upper, axis=0)
        return self

    def transform(self, X):
        X = np.asarray(X, dtype=float)
        return np.clip(X, self.lower_bounds_, self.upper_bounds_)

# =============================
# 4. Apply Winsorization to train and test
# =============================
winsor_full = Winsorizer(lower=0.5, upper=99.5)
winsor_full.fit(X_full_raw)

X_full_win = winsor_full.transform(X_full_raw)
X_test_win = winsor_full.transform(X_test_raw)

print("X_full_win shape:", X_full_win.shape)
print("X_test_win shape:", X_test_win.shape)


# =============================
# 5. KNN imputation on winsorized data
# =============================
knn_imputer = KNNImputer(
    n_neighbors=5,
    weights="uniform"
)

# fit on winsorized training data only
knn_imputer.fit(X_full_win)

# transform train and test
X_full_imp = knn_imputer.transform(X_full_win)
X_test_imp = knn_imputer.transform(X_test_win)

print("Any NaNs left in training after KNN imputation?",
      np.isnan(X_full_imp).any())
print("Any NaNs left in test after KNN imputation?",
      np.isnan(X_test_imp).any())

Train shape: (10000, 65)
Test shape: (8000, 65)
X_full_win shape: (10000, 64)
X_test_win shape: (8000, 64)
Any NaNs left in training after KNN imputation? False
Any NaNs left in test after KNN imputation? False


# XGBoost (hist = 0.906)


In [ ]:
# 5. Define XGBoost model (logistic objective)
xgb_raw = XGBClassifier(
    objective="binary:logistic",
    eval_metric="auc",
    n_estimators=1000,
    max_depth=6,
    learning_rate=0.02,
    subsample=0.9,
    min_child_weight=1,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    reg_alpha=0.1,
    tree_method="exact", #hist
    scale_pos_weight=1,
    n_jobs=-1
)

# 6. Cross validation AUC
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_auc_scores = cross_val_score(
    xgb_raw,
    X_full_raw, # raw
    y_full,
    cv=skf,
    scoring="roc_auc"
)

print("XGBoost CV AUC scores:", cv_auc_scores)
print("XGBoost mean CV AUC:", cv_auc_scores.mean())

# =============================
# 7. Fit final model on full training data
# =============================
xgb_raw.fit(X_full_raw, y_full)

# =============================
# 8. Predict probabilities for test set
# =============================
# predict probability of class 1
test_pred_proba = xgb_raw.predict_proba(X_test_raw)[:, 1]

XGBoost CV AUC scores: [0.91790203 0.93594137 0.89631165 0.9116121  0.87460146]
XGBoost mean CV AUC: 0.9072737233411697


In [ ]:
# =============================
# 9. Create submission file
# =============================

# Option 2: build directly if you know the id column name
# Replace "ID" with your actual id column
submission = pd.DataFrame({
    "ID": test["ID"],
    "class": test_pred_proba
})
submission.to_csv("exact_weight1_win.csv", index=False)